# EA2 - Actividad 2.6: Visualizacion de Datos

## Objetivos
- Convertir resultados de Spark a Pandas para visualizar
- Crear graficos de barras, histogramas y lineas con matplotlib
- Usar seaborn para heatmaps y boxplots
- Combinar multiples graficos en dashboards con subplots

## Conceptos Clave

### De Spark a Pandas: `toPandas()`

Spark procesa datos de forma distribuida, pero las librerias de visualizacion (matplotlib, seaborn) trabajan con datos locales en memoria. El metodo `toPandas()` convierte un DataFrame de Spark a un DataFrame de Pandas.

**Importante:** `toPandas()` trae **todos los datos al driver** (memoria local). Usarlo solo con:
- Resultados agregados (pocos registros)
- Subconjuntos pequenos de datos
- **NUNCA** con millones de registros sin agregar primero

### Tipos de Graficos

| Tipo | Uso | Libreria |
|------|-----|----------|
| **Barras** | Comparar categorias | matplotlib |
| **Histograma** | Distribucion de una variable | matplotlib |
| **Lineas** | Tendencias temporales | matplotlib |
| **Heatmap** | Correlaciones entre variables | seaborn |
| **Boxplot** | Distribucion y outliers | seaborn |
| **Subplots** | Dashboard con multiples graficos | matplotlib |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder.appName("visualizacion").master("local[*]").getOrCreate()

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Spark version: {spark.version}")

## 1. Cargar Datos

Usaremos los datasets de ventas (sales.csv) y tiendas (stores.csv), combinandolos con un join.

In [ ]:
# Leer datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

# Combinar con join
df = df_sales.join(df_stores, "Store")

print(f"Registros de ventas: {df_sales.count()}")
print(f"Tiendas: {df_stores.count()}")
print(f"Registros combinados: {df.count()}")
df.printSchema()

In [ ]:
# Vista previa de los datos
df.show(5)

## 2. Conversion a Pandas con `toPandas()`

> **Advertencia:** `toPandas()` trae todos los datos al driver local. Siempre agrega los datos **antes** de convertir para evitar problemas de memoria. Nunca uses `toPandas()` directamente sobre un DataFrame con millones de filas.

In [ ]:
# CORRECTO: Agregar primero, luego convertir (pocos registros)
df_tipo = df.groupBy("Type").agg(
    F.sum("Weekly_Sales").alias("total")
).toPandas()

print(f"Registros en Pandas: {len(df_tipo)}")
print(df_tipo)

## 3. Grafico de Barras con Matplotlib

Los graficos de barras son ideales para comparar valores entre categorias.

In [ ]:
# Grafico de barras: Ventas totales por tipo de tienda
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_tipo["Type"], df_tipo["total"], color=["#2196F3", "#4CAF50", "#FF9800"])
ax.set_title("Ventas Totales por Tipo de Tienda", fontsize=14)
ax.set_xlabel("Tipo")
ax.set_ylabel("Ventas Totales ($)")
plt.tight_layout()
plt.show()

## 4. Histograma de Distribucion

Los histogramas muestran como se distribuyen los valores de una variable numerica.

In [ ]:
# Preparar datos: muestra de ventas semanales
df_hist = df.select("Weekly_Sales").filter(F.col("Weekly_Sales") > 0) \
    .sample(fraction=0.1, seed=42).toPandas()

# Histograma de distribucion de ventas
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_hist["Weekly_Sales"], bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
ax.set_title("Distribucion de Ventas Semanales", fontsize=14)
ax.set_xlabel("Ventas Semanales ($)")
ax.set_ylabel("Frecuencia")
ax.axvline(df_hist["Weekly_Sales"].mean(), color="red", linestyle="--", label=f"Media: ${df_hist['Weekly_Sales'].mean():,.0f}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Grafico de Lineas (Serie Temporal)

Los graficos de lineas son perfectos para visualizar tendencias a lo largo del tiempo.

In [ ]:
# Preparar datos: ventas promedio por fecha
df_temporal = df.groupBy("Date").agg(
    F.avg("Weekly_Sales").alias("avg_sales")
).orderBy("Date").toPandas()

# Convertir fecha a datetime
df_temporal["Date"] = pd.to_datetime(df_temporal["Date"])

# Grafico de lineas
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_temporal["Date"], df_temporal["avg_sales"], color="#2196F3", linewidth=1.5)
ax.set_title("Tendencia de Ventas Promedio Semanales", fontsize=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas Promedio ($)")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Seaborn: Heatmap de Correlacion

El heatmap muestra la correlacion entre variables numericas. Valores cercanos a 1 o -1 indican correlacion fuerte.

In [ ]:
# Preparar datos para correlacion
df_corr = df.select(
    "Weekly_Sales", "Size", "Temperature", "Fuel_Price", "CPI", "Unemployment"
).toPandas()

# Heatmap de correlacion
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, cmap="coolwarm", ax=ax, 
            fmt=".2f", linewidths=0.5, center=0)
plt.title("Matriz de Correlacion", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Seaborn: Boxplot

Los boxplots muestran la distribucion de una variable, incluyendo la mediana, cuartiles y valores atipicos (outliers).

In [ ]:
# Preparar datos para boxplot
df_box = df.select("Type", "Weekly_Sales") \
    .filter(F.col("Weekly_Sales").between(0, 100000)) \
    .sample(fraction=0.1, seed=42).toPandas()

# Boxplot de ventas por tipo de tienda
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df_box, x="Type", y="Weekly_Sales", palette="Set2", ax=ax)
ax.set_title("Distribucion de Ventas por Tipo de Tienda", fontsize=14)
ax.set_xlabel("Tipo de Tienda")
ax.set_ylabel("Ventas Semanales ($)")
plt.tight_layout()
plt.show()

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Top 10 departamentos por ventas totales
# =============================================================
# TODO: Crea un grafico de barras horizontales con los top 10
#   departamentos (Dept) por ventas totales.
#
# Pasos:
#   1. Agregar ventas por departamento:
#      df_dept = df.groupBy("Dept").agg(
#          F.sum("Weekly_Sales").alias("total")
#      ).orderBy(F.desc("total")).limit(10).toPandas()
#
#   2. Crear grafico de barras HORIZONTALES:
#      fig, ax = plt.subplots(figsize=(10, 6))
#      ax.barh(df_dept["Dept"].astype(str), df_dept["total"])
#
#   3. Agregar titulo: "Top 10 Departamentos por Ventas Totales"
#      xlabel: "Ventas Totales ($)"
#      ylabel: "Departamento"
#
# Pista: Usa ax.barh() en vez de ax.bar() para barras horizontales

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Boxplot de ventas por tipo de tienda
# =============================================================
# TODO: Crea un boxplot comparando la distribucion de
#   Weekly_Sales por Type de tienda (A, B, C).
#
# Pasos:
#   1. Preparar datos (filtrar outliers extremos y tomar muestra):
#      df_box2 = df.select("Type", "Weekly_Sales") \
#          .filter(F.col("Weekly_Sales").between(0, 80000)) \
#          .sample(fraction=0.1, seed=42).toPandas()
#
#   2. Crear boxplot con seaborn:
#      fig, ax = plt.subplots(figsize=(8, 6))
#      sns.boxplot(data=df_box2, x="Type", y="Weekly_Sales", ...)
#
#   3. Agregar titulo: "Distribucion de Ventas Semanales por Tipo"
#
# Pista: Experimenta con el parametro palette ("Set1", "Set2", "pastel")

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Tendencia mensual de ventas
# =============================================================
# TODO: Crea un grafico de lineas con la tendencia mensual
#   de ventas (agregando por mes).
#
# Pasos:
#   1. Extraer anio y mes de la columna Date:
#      df_mensual = df.withColumn("anio", F.year("Date")) \
#          .withColumn("mes", F.month("Date"))
#
#   2. Agregar ventas por anio y mes:
#      df_mensual = df_mensual.groupBy("anio", "mes").agg(
#          F.sum("Weekly_Sales").alias("total")
#      ).orderBy("anio", "mes").toPandas()
#
#   3. Crear columna de fecha para el eje X:
#      df_mensual["fecha"] = pd.to_datetime(
#          df_mensual["anio"].astype(str) + "-" + df_mensual["mes"].astype(str) + "-01"
#      )
#
#   4. Crear grafico de lineas:
#      fig, ax = plt.subplots(figsize=(12, 5))
#      ax.plot(df_mensual["fecha"], df_mensual["total"], marker="o")
#
#   5. Agregar titulo: "Tendencia Mensual de Ventas"
#
# Pista: Usa marker="o" para mostrar puntos en cada mes

# Escribe tu codigo aqui:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Dashboard con multiples graficos en una sola figura.

In [ ]:
# =============================================================
# DESAFIO: Dashboard con 4 graficos usando subplots(2,2)
# =============================================================
# TODO: Crea una figura con 4 graficos organizados en 2x2:
#
#   Posicion (0,0): Barras - Ventas por tipo de tienda
#   Posicion (0,1): Linea - Serie temporal de ventas
#   Posicion (1,0): Heatmap - Correlacion entre variables
#   Posicion (1,1): Boxplot - Ventas por tipo de tienda
#
# Pasos:
#   1. Preparar los 4 DataFrames de Pandas (reusar los de arriba)
#
#   2. Crear figura con subplots:
#      fig, axes = plt.subplots(2, 2, figsize=(16, 12))
#
#   3. Grafico (0,0) - Barras:
#      axes[0, 0].bar(...)
#      axes[0, 0].set_title("Ventas por Tipo de Tienda")
#
#   4. Grafico (0,1) - Linea temporal:
#      axes[0, 1].plot(...)
#      axes[0, 1].set_title("Tendencia Temporal")
#
#   5. Grafico (1,0) - Heatmap:
#      sns.heatmap(..., ax=axes[1, 0])
#      axes[1, 0].set_title("Correlacion")
#
#   6. Grafico (1,1) - Boxplot:
#      sns.boxplot(..., ax=axes[1, 1])
#      axes[1, 1].set_title("Distribucion por Tipo")
#
#   7. Titulo general:
#      fig.suptitle("Dashboard de Ventas", fontsize=16, fontweight="bold")
#      plt.tight_layout()
#      plt.show()
#
# Pista: Cada subplot se accede con axes[fila, columna]

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **toPandas():** Convertir DataFrames de Spark a Pandas (solo con datos agregados/pequenos)
2. **Graficos de barras:** `ax.bar()` y `ax.barh()` para comparar categorias
3. **Histogramas:** `ax.hist()` para ver distribuciones de variables
4. **Graficos de lineas:** `ax.plot()` para tendencias temporales
5. **Heatmap:** `sns.heatmap()` para visualizar correlaciones
6. **Boxplot:** `sns.boxplot()` para distribucion y outliers
7. **Subplots:** `plt.subplots(filas, columnas)` para dashboards con multiples graficos
8. **Buenas practicas:** Siempre agregar datos antes de `toPandas()` para evitar problemas de memoria

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")